# 08 · Table 3: performance restricted to supplement studies and to meal studies

Produces the **"Restricted to iron supplement studies"** and **"Restricted to meal studies"** rows of
**Table 3**, with the same procedure as notebook 01 (same predictors and encoding, 80:20 split, grid
search with five-fold cross-validation in the training set, held-out test metrics with bootstrap 95% CIs,
five-fold cross-validation in the training set). The all-studies row is recomputed as a check.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, make_scorer

RANDOM_STATE = 42
PARAM_GRID = {"n_estimators": [300, 400], "learning_rate": [0.01, 0.05], "max_depth": [8, 9], "subsample": [0.8, 0.9], "colsample_bytree": [0.5, 0.6, 0.7]}
os.makedirs("../results", exist_ok=True)

## 1. Data (preprocessing as in notebook 01, before restricting by administration route)

In [2]:
raw = pd.read_excel("../data/pooled_dataset.xls")
route = raw["Meal or Supplement"].copy()
d = raw.drop(columns=["Order", "Study", "Gender", "Idold", "BW", "BH", "Alpha glycoprotein", "Site",
         "Meal or Supplement", "Meat protein", "Milk protein", "Tea dose", "MC.last",
         "MC.freq", "MC.duration", "MC.percentage", "GOS dose ", "Ethnicity", "sTfR"])
d["Meal type"] = d["Meal type"].fillna("Supplement")
d["Isotope"] = d["Isotope"].fillna("Unknown")
for col in d.columns:
    if d[col].dtype != float:
        d[col] = d[col].astype("category").cat.codes
d = d[d.isnull().sum(axis=1) < 14]
route = route.loc[d.index]
print(route.value_counts())

Meal or Supplement
Meal          3780
Supplement    1278
Name: count, dtype: int64


## 2. Table 3

In [3]:
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps))


def bootstrap_ci(y_true, y_hat, metric_fn, B=2000, random_state=RANDOM_STATE):
    """Percentile bootstrap 95% CI of a test-set metric (model held fixed)."""
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true); y_hat = np.asarray(y_hat)
    n = len(y_true); idx = np.arange(n); stats = np.empty(B)
    for b in range(B):
        bs = rng.choice(idx, size=n, replace=True)
        stats[b] = metric_fn(y_true[bs], y_hat[bs])
    return np.percentile(stats, [2.5, 97.5])

smape_scorer = make_scorer(smape, greater_is_better=False)


def run_subset(df, label):
    y = np.log10(df["Fe absorption (%)"] + 1)
    X = df[[c for c in df.columns if c != "Fe absorption (%)"]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

    gs = GridSearchCV(xgb.XGBRegressor(objective="reg:squarederror"), PARAM_GRID, cv=5,
                      scoring="neg_mean_squared_error", n_jobs=-1)
    gs.fit(X_train, y_train)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1, **gs.best_params_)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = {
        "RMSE": -cross_val_score(model, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1),
        "MAE": -cross_val_score(model, X_train, y_train, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1),
        "R²": cross_val_score(model, X_train, y_train, scoring="r2", cv=cv, n_jobs=-1),
        "sMAPE": -cross_val_score(model, X_train, y_train, scoring=smape_scorer, cv=cv, n_jobs=-1),
    }
    rows = []
    for name, fn in [("RMSE", root_mean_squared_error), ("MAE", mean_absolute_error), ("R²", r2_score), ("sMAPE", smape)]:
        lo, hi = bootstrap_ci(y_test, y_pred, fn)
        rows.append({"Studies": label, "n (training/test)": f"{len(X_train)}/{len(X_test)}", "Metric": name,
                     "Test value": fn(y_test, y_pred), "95% CI lower": lo, "95% CI upper": hi,
                     "CV mean": cv_scores[name].mean(), "CV SD": cv_scores[name].std(ddof=1)})
    print(label, "| selected hyperparameters:", gs.best_params_)
    return pd.DataFrame(rows)


table3 = pd.concat([
    run_subset(d, "All studies"),
    run_subset(d[route == "Supplement"], "Restricted to iron supplement studies"),
    run_subset(d[route == "Meal"], "Restricted to meal studies"),
], ignore_index=True).round(3)
print()
print(table3.to_string(index=False))
table3.to_csv("../results/table3.csv", index=False)

All studies | selected hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 400, 'subsample': 0.8}


Restricted to iron supplement studies | selected hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 300, 'subsample': 0.8}


Restricted to meal studies | selected hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 400, 'subsample': 0.9}

                              Studies n (training/test) Metric  Test value  95% CI lower  95% CI upper  CV mean  CV SD
                          All studies         4046/1012   RMSE       0.206         0.195         0.219    0.219  0.011
                          All studies         4046/1012    MAE       0.160         0.152         0.168    0.169  0.008
                          All studies         4046/1012     R²       0.757         0.724         0.783    0.722  0.028
                          All studies         4046/1012  sMAPE       0.200         0.187         0.213    0.207  0.009
Restricted to iron supplement studies          1022/256   RMSE       0.186         0.164         0.209    0.205  0.011
Restricted to iron supplement studies          1022/256    MAE       0.141         0.126         0.156    0.155  0.008
Restric